# Panel method for NACA 4-digit airfoils

Walkthrough of the study with the `panelfoil` package: the three-panel hand calculation, validation of
NACA 3217 against XFOIL, the mesh study, the inverse design (+20 % lift at α = 0) and the viscous
comparison of the redesigned NACA 4114. Every number comes from `src/panelfoil/`, covered by the tests.

Lucas García Franco and Carlos Pesquero Cañizares, ETSIAE-UPM.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: install the package (code + XFOIL data) from GitHub
    %pip install -q git+https://github.com/CPesquero/panel-method-airfoil-design.git

import numpy as np

import panelfoil as pf
from panelfoil import plots

np.set_printoptions(precision=3, suppress=True)

## 1. Hand calculation: three-panel wedge

Wedge from the ID digits (X = 7, Y = 3, Z = 8): α = 3.1°, vertex at (0.4, 0.18). Panels run clockwise
from the trailing edge, so every normal points out of the body. Unknowns: three source strengths and
one vortex strength γ; the fourth equation is the Kutta condition between panels 1 and 3.

In [ ]:
Xw, Yw = pf.wedge()
plots.wedge(Xw, Yw)
A, I = pf.system(Xw, Yw, pf.ALPHA_DEG)
print("A =", A, sep="\n")
print("I =", I)
sw = pf.solve(Xw, Yw, pf.ALPHA_DEG)
print(f"gamma = {sw.gamma:.3f}   Cl = {sw.cl:.3f}")

## 2. NACA 3217 against XFOIL (inviscid), N = 100

Lift from the circulation (Kutta–Joukowski), moment from the surface pressures about c/4, positive
nose-up. A cambered section gives a nose-down (negative) moment, as XFOIL does.

In [ ]:
ref = pf.REF_NACA3217
cp_xfoil = pf.load_cp()
s100 = pf.solve(*pf.BASELINE.coordinates(100), pf.ALPHA_DEG)
print(f"Cl     = {s100.cl:.4f}  (XFOIL {ref['cl']})")
print(f"Cm_c/4 = {s100.cm_c4:.4f}  (XFOIL {ref['cm_c4']})")
plots.cp_vs_xfoil(s100, cp_xfoil, "NACA 3217, α = 3.1°, N = 100");

## 3. Mesh convergence

Start at 40 panels and grow 20 % per step; N_opt is the first mesh that changes Cl by less than 1 %.
Note that Cl peaks near N ≈ 82 and then drifts down slightly, so the convergence is not monotonic.

In [ ]:
conv = pf.mesh_convergence()
for n, cl in zip(conv.n_panels, conv.cl):
    print(f"N = {n:3d}   Cl = {cl:.5f}")
print("N_opt =", conv.n_opt)
plots.convergence(conv);

In [ ]:
s_opt = pf.solve(*pf.BASELINE.coordinates(conv.n_opt), pf.ALPHA_DEG)
print(f"N = {conv.n_opt}:  Cl = {s_opt.cl:.4f}   Cm_c/4 = {s_opt.cm_c4:.4f}")
plots.cp_vs_xfoil(s_opt, cp_xfoil, f"NACA 3217, α = 3.1°, N = {conv.n_opt}");

## 4. Inverse design: +20 % Cl at α = 0

Grid search over f = 1–9 %, p = 10–60 %, t = 5–25 %. Sections whose Cl is within 0.015 of the target
are kept, and the one whose moment changes least wins.

In [ ]:
d = pf.inverse_design(n_panels=conv.n_opt)
print(f"base  {d.base.airfoil}: Cl = {d.base.cl:.4f}   Cm = {d.base.cm_c4:.4f}")
print(f"best  {d.best.airfoil}: Cl = {d.best.cl:.4f} ({d.cl_gain:+.1%})   Cm = {d.best.cm_c4:.4f}")
print(f"{len(d.feasible)} sections met the lift target")
plots.design_geometry(d)
plots.design_cp(d, 100);

## 5. NACA 4114: inviscid vs viscous (XFOIL, Re = 3·10⁶)

The panel method (markers) follows XFOIL's inviscid lift closely. Viscosity lowers the slope and the
section stalls at Cl ≈ 2.0, α = 17°.

In [ ]:
inv = pf.load_polar(pf.POLAR_INVISCID)
vis = pf.load_polar(pf.POLAR_VISCOUS)
al = np.arange(-4, 13, 2.0)
pan = [pf.solve(*pf.OPTIMIZED.coordinates(conv.n_opt), a) for a in al]
plots.polars(inv, vis, (al, [p.cl for p in pan], [p.cm_c4 for p in pan]))
print("viscous Cl_max = {1:.3f} at alpha = {0:g} deg".format(*vis.cl_max))